# **Speech Commands V2 - 12 Class benchmark**

In [ ]:
import os

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import (
    Conv1D, Dense, Flatten, Dropout, MaxPooling1D,
    BatchNormalization, GlobalAveragePooling1D
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

import tensorflow_model_optimization as tfmot  # For QAT
import keras_tuner as kt

import larq
from larq.layers import QuantConv1D, QuantDense
from larq.models import summary as larq_summary
import larq_compute_engine as lce

import numpy as np
import pandas as pd
import librosa  # For MFCCs
import io       # For reading audio bytes with librosa
import random
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

import requests
import urllib.request
import tarfile
from tqdm.notebook import tqdm
from tabulate import tabulate

In [ ]:
# Check if GPU is Available
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        print("GPU:", gpu.name)
else:
    print("No GPU devices found.")

## Dataset Preparation

In [ ]:
import requests
from tqdm.notebook import tqdm
import soundfile as sf 

# --- Configuration ---
dataset_url = "http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz"
root_dir = "./google_dataset_v2"
archive_path = os.path.join(root_dir, "speech_commands_v0.02.tar.gz")
extracted_dir = os.path.join(root_dir, "google_speech_recognition_v2")

os.makedirs(root_dir, exist_ok=True)

if os.path.exists(extracted_dir) and os.path.isdir(extracted_dir):
    print(f"Dataset already extracted at: {extracted_dir}")
else:
    if not os.path.exists(archive_path):
        print(f"Downloading dataset to: {archive_path}")
        response = requests.get(dataset_url, stream=True)
        total_size = int(response.headers.get('content-length', 0))
        block_size = 1024

        with open(archive_path, 'wb') as f, tqdm(
            desc="Downloading SpeechCommands v2",
            total=total_size,
            unit='iB',
            unit_scale=True
        ) as bar:
            for data in response.iter_content(block_size):
                f.write(data)
                bar.update(len(data))
        print("Download complete.")
    else:
        print(f"Archive already exists at: {archive_path}")

    print(f"Extracting to: {extracted_dir}")
    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall(path=extracted_dir)
    print("Extraction complete.")

In [ ]:
# --- Define 12-class label set ---
CORE_LABELS = ['yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go']
ALL_LABELS = CORE_LABELS + ['_silence_', '_unknown_']

def load_data():
    X_raw = []
    y_raw = []
    classes = {label: idx for idx, label in enumerate(ALL_LABELS)}

    target_sample_rate = 16000
    target_length_samples = 16000

    # MFCC Parameters
    n_mfcc = 13
    n_fft = 400
    hop_length = 160
    max_len = int(np.ceil(target_length_samples / hop_length)) + 1
    print(f"Using MFCC parameters: n_mfcc={n_mfcc}, n_fft={n_fft}, hop_length={hop_length}, max_len={max_len}")

    dataset_path = "./google_dataset_v2/google_speech_recognition_v2"

    silence_dir = os.path.join(dataset_path, "_background_noise_")
    silence_files = tf.io.gfile.glob(os.path.join(silence_dir, "*.wav"))
    for file_path in silence_files:
        try:
            wav_data, sr = librosa.load(file_path, sr=target_sample_rate)
            for i in range(10): # Generate 10 1-second chunks
                start = random.randint(0, len(wav_data) - target_length_samples - 1)
                chunk = wav_data[start:start + target_length_samples]
                wav_bytes = io.BytesIO()
                sf.write(wav_bytes, chunk, sr, format='WAV')
                X_raw.append(wav_bytes.getvalue())
                y_raw.append("_silence_")
        except Exception as e:
            print(f"Error processing silence file {file_path}: {e}")

    all_classes = [d for d in tf.io.gfile.listdir(dataset_path) if tf.io.gfile.isdir(os.path.join(dataset_path, d))]
    for folder in all_classes:
        if folder.startswith("_"):
            continue
        class_path = os.path.join(dataset_path, folder)
        class_files = tf.io.gfile.listdir(class_path)
        for file_name in class_files:
            file_path = f"{class_path}/{file_name}"
            try:
                with tf.io.gfile.GFile(file_path, 'rb') as file:
                    wav_data_bytes = file.read()
                    label = folder if folder in CORE_LABELS else "_unknown_"
                    X_raw.append(wav_data_bytes)
                    y_raw.append(label)
            except Exception as e:
                print(f"Warning: Could not read file {file_path}: {e}")

    print(f"Total loaded samples: {len(X_raw)}")

    X_train_raw, X_temp_raw, y_train_raw, y_temp_raw = train_test_split(
        X_raw, y_raw, test_size=0.15, random_state=42, stratify=y_raw
    )
    X_val_raw, X_test_raw, y_val_raw, y_test_raw = train_test_split(
        X_temp_raw, y_temp_raw, test_size=0.33, random_state=42, stratify=y_temp_raw
    )

    def process_audio_to_mfcc(audio_bytes_list, labels_list):
        features = []
        labels = []
        for audio_bytes, label in zip(audio_bytes_list, labels_list):
            try:
                wav_data, sample_rate = librosa.load(io.BytesIO(audio_bytes), sr=target_sample_rate)
                if len(wav_data) > target_length_samples:
                    wav_data = wav_data[:target_length_samples]
                elif len(wav_data) < target_length_samples:
                    wav_data = np.pad(wav_data, (0, target_length_samples - len(wav_data)), 'constant')

                mfccs = librosa.feature.mfcc(y=wav_data, sr=sample_rate, n_mfcc=n_mfcc,
                                             n_fft=n_fft, hop_length=hop_length)
                delta = librosa.feature.delta(mfccs)
                delta2 = librosa.feature.delta(mfccs, order=2)
                mfcc_full = np.concatenate([mfccs, delta, delta2], axis=0).T 

                if mfcc_full.shape[0] > max_len:
                    mfcc_full = mfcc_full[:max_len]
                elif mfcc_full.shape[0] < max_len:
                    pad_width = max_len - mfcc_full.shape[0]
                    mfcc_full = np.pad(mfcc_full, ((0, pad_width), (0, 0)), mode='constant')

                features.append(mfcc_full)
                labels.append(classes[label])
            except Exception as e:
                print(f"Error processing audio for label {label}: {e}")

        return np.array(features), np.array(labels)

    print("Processing training data...")
    X_train, y_train = process_audio_to_mfcc(X_train_raw, y_train_raw)
    print("Processing validation data...")
    X_val, y_val = process_audio_to_mfcc(X_val_raw, y_val_raw)
    print("Processing test data...")
    X_test, y_test = process_audio_to_mfcc(X_test_raw, y_test_raw)

    input_shape = X_train.shape[1:]

    return X_train, X_val, X_test, y_train, y_val, y_test, input_shape, classes

X_train, X_val, X_test, y_train, y_val, y_test, input_shape_mfcc, classes = load_data()

print("\n--- Final Shapes ---")
print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)
print("Input shape for model:", input_shape_mfcc)
print("Classes:", classes)

#### Plot data samples

In [ ]:
import librosa.display

def plot_mfcc_samples(X, y, classes_map_dict, n=5, sr=16000, hop_length=160):
    fig, axes = plt.subplots(n, 1, figsize=(10, 4 * n))
    if n == 1:
        axes = [axes]
        
    label_names = {v: k for k, v in classes_map_dict.items()}

    for i in range(n):
        index = random.randint(0, len(X) - 1)
        mfcc_data = X[index]
        label_int = y[index]
        label_str = label_names.get(label_int, "Unknown")

        img = librosa.display.specshow(mfcc_data.T,
                                       sr=sr,
                                       hop_length=hop_length,
                                       x_axis='time',
                                       ax=axes[i])
        fig.colorbar(img, ax=axes[i], format='%+2.0f dB')
        axes[i].set_title(f"MFCC (+Deltas) for Label: '{label_str}' (Index: {index})")

    plt.tight_layout()
    plt.show()

if 'classes' in locals():
    plot_mfcc_samples(X_train, y_train, classes, n=5)
else:
    print("Cannot plot samples because 'classes' is not defined.")

## NN Evaluation Methods

In [ ]:
def evaluate_NN_models(model_list, X_test, y_test, bin_class=False):
    loss, accuracy, y_pred, precision, recall, f1_score, support, confusion_matrix_list = [], [], [], [], [], [], [], []
    for model in model_list:
        loss_value, accuracy_value = model.evaluate(X_test, y_test)
        loss.append(loss_value)
        accuracy.append(accuracy_value)
        if bin_class:
            y_pred_value = (model.predict(X_test) > 0.5).astype(int)
        else:
            y_pred_value = np.argmax(model.predict(X_test), axis=1)
        y_pred.append(y_pred_value)
        y_test_classes = y_test
        precision_value, recall_value, f1_score_value, support_value = precision_recall_fscore_support(
            y_test_classes, 
            y_pred_value, 
            average='binary' if bin_class else 'weighted',
            zero_division=0
        )
        precision.append(precision_value)
        recall.append(recall_value)
        f1_score.append(f1_score_value)
        support.append(support_value)
        confusion_matrix_value = confusion_matrix(y_test_classes, y_pred_value, labels=np.arange(len(classes)))
        confusion_matrix_list.append(confusion_matrix_value)
    return [accuracy, precision, recall, f1_score], confusion_matrix_list, y_pred

In [ ]:
# --- print_table Function ---
def print_table(data, headers):
    table = tabulate(data, headers=headers, tablefmt='grid')
    print(table)

In [ ]:
def plot_confusion_matrix(ax, conf_matrix, title, cmap):
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap=cmap, cbar=False, ax=ax)
    ax.set_title(title)

In [ ]:
def draw_confusion_matrix(matrix,name):
    colormap=["Blues","Greens","Oranges"]
    fig, axes = plt.subplots(1, 1, figsize=(15, 5))
    for i in range(len(matrix)):
        plot_confusion_matrix(axes, matrix[i], "Confusion Matrix  ", colormap[i])
    plt.tight_layout()

In [ ]:
# Define early stopping callback (primarily for tuner)
early_stopping_tuner = EarlyStopping(
    monitor='val_loss',
    patience=25,
    verbose=1,
    mode='min',
    restore_best_weights=True
)
print("Early stopping callback for tuner defined.")

## Hyperparameters Optimization

In [ ]:
class BinaryNetHyperModel(kt.HyperModel):
    def __init__(self, input_shape_mfcc, num_classes):
        self.input_shape = input_shape_mfcc
        self.num_classes = num_classes

    def build(self, hp):
        kwargs = dict(input_quantizer="ste_sign",
                     kernel_quantizer="ste_sign",
                     kernel_constraint="weight_clip")

        # --- Choose kernel size for this trial ---
        kernel_choice = hp.Choice('kernel_size', values=[3, 5])

        model = Sequential()

        # Tune the first Conv1D layer
        filters_conv1 = hp.Int('conv1_filters', min_value=32, max_value=128, step=32) # Increased range
        model.add(Conv1D(filters=filters_conv1, kernel_size=kernel_choice, activation='relu', # Tuned kernel
                        input_shape=self.input_shape))
        model.add(MaxPooling1D(pool_size=2))
        model.add(BatchNormalization(scale=False))

        # Tune the first QuantConv1D layer
        filters_qconv1 = hp.Int('qconv1_filters', min_value=8, max_value=32, step=8) # Increased range
        model.add(QuantConv1D(filters_qconv1, kernel_size=kernel_choice, # Tuned kernel
                             kernel_quantizer="ste_sign",
                             kernel_constraint="weight_clip",
                             use_bias=False))
        model.add(BatchNormalization(scale=False))
        model.add(Dropout(rate=hp.Float('dropout1', min_value=0.2, max_value=0.5)))

        # Tune the second QuantConv1D layer
        filters_qconv2 = hp.Int('qconv2_filters', min_value=8, max_value=32, step=8) # Increased range
        model.add(QuantConv1D(filters_qconv2, kernel_size=kernel_choice, use_bias=False, **kwargs)) # Tuned kernel
        model.add(BatchNormalization(scale=False))
        model.add(Dropout(rate=hp.Float('dropout2', min_value=0.2, max_value=0.5)))

        model.add(tf.keras.layers.GlobalAveragePooling1D())

        # Tune the QuantDense layer
        units_qdense = hp.Int('qdense_units', min_value=8, max_value=32, step=8) # Increased range
        model.add(QuantDense(units_qdense, use_bias=False, **kwargs))
        model.add(BatchNormalization(scale=False))
        model.add(Dense(self.num_classes, activation='softmax'))

        learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')
        model.compile(
            optimizer=Adam(learning_rate=learning_rate),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        return model

In [ ]:
# Initialize the hypermodel with the MFCC+delta shape
hypermodel = BinaryNetHyperModel(input_shape_mfcc=input_shape_mfcc, num_classes=len(classes))

# Create the tuner
tuner = kt.BayesianOptimization(
    hypermodel,
    objective='val_accuracy',
    max_trials=20, 
    directory='bayesian_opt',
    project_name='sc_tuning_bnn'
)

# Search for the best hyperparameters
tuner.search(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping_tuner]
)

# Get the best hyperparameters
best_hps_list = tuner.get_best_hyperparameters(num_trials=3)
best_models_list = tuner.get_best_models(num_models=3)

In [ ]:
# Show the top 3 models
print("\nTop 3 Models Performance:")
print("-" * 50)

for idx, hps in enumerate(best_hps_list, 1):
    print(f"\nModel {idx}:")
    print("Hyperparameters:")
    for param, value in hps.values.items():
        print(f"{param}: {value}")
    print(f"Validation Accuracy: {best_models_list[idx-1].evaluate(X_val, y_val, verbose=0)[1]:.3f}")
        
# Let user choose the model
while True:
    try:
        choice = int(input("\nChoose a model (1, 2, or 3): "))
        if 1 <= choice <= 3:
            break
        print("Please enter 1, 2, or 3")
    except ValueError:
        print("Please enter a valid number")

# Use the chosen model
# model = best_models_list[choice - 1]
chosen_hps = best_hps_list[choice - 1]

print(f"\nSelected Model {choice}")
print("Final hyperparameters:")
for param, value in chosen_hps.values.items():
    print(f"{param}: {value}")

## Build BNN

In [ ]:
# --- Final Training Parameters ---

# Use LR from the chosen hyperparameters
final_lr = chosen_hps.get('learning_rate')
final_epochs = 500
final_batch_size = 1024 

print(f"Final Training LR: {final_lr:.6f}")
print(f"Final Training Max Epochs: {final_epochs}")
print(f"Final Training Batch Size: {final_batch_size}")

# Define EarlyStopping for final training runs (monitor val_accuracy)
final_early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=30, # Can use slightly more patience for final run
    verbose=1,
    mode='max',
    restore_best_weights=True
)
print("Early stopping for final training defined.")

In [ ]:
# --- Build BNN Model (Using Tuned HPs) ---
print(f"Using input shape: {input_shape_mfcc}")
print(f"Using hyperparameters: {chosen_hps.values}")

# All quantized layers except the first will use the same options
kwargs = dict(input_quantizer="ste_sign",
            kernel_quantizer="ste_sign",
            kernel_constraint=larq.constraints.WeightClip(1.3))

# Get the kernel size chosen by the BNN tuner
kernel_choice = chosen_hps.get('kernel_size')

# Define the BNN model structure
model_bnn = Sequential(name="BNN_MFCC_Tuned") # Specific name for BNN model

# First Conv1D layer (Full Precision) with tuned filters/kernel
model_bnn.add(Conv1D(filters=chosen_hps.get('conv1_filters'), kernel_size=kernel_choice, activation='relu',
                 input_shape=input_shape_mfcc)) # Use MFCC shape
model_bnn.add(MaxPooling1D(pool_size=2))
model_bnn.add(BatchNormalization(scale=False))

# First QuantConv1D layer with tuned filters/kernel
model_bnn.add(QuantConv1D(chosen_hps.get('qconv1_filters'), kernel_size=kernel_choice,
                      kernel_quantizer="ste_sign",
                      kernel_constraint=larq.constraints.WeightClip(1.3),
                      use_bias=False))
model_bnn.add(BatchNormalization(scale=False))
model_bnn.add(Dropout(rate=chosen_hps.get('dropout1')))

# Second QuantConv1D layer with tuned filters/kernel
model_bnn.add(QuantConv1D(chosen_hps.get('qconv2_filters'), kernel_size=kernel_choice, use_bias=False, **kwargs))
model_bnn.add(BatchNormalization(scale=False))
model_bnn.add(Dropout(rate=chosen_hps.get('dropout2')))

model_bnn.add(tf.keras.layers.GlobalAveragePooling1D())

# QuantDense layer with tuned units
model_bnn.add(QuantDense(chosen_hps.get('qdense_units'), use_bias=False, **kwargs)) # kwargs used here too
model_bnn.add(BatchNormalization(scale=False))

# Final output layer (Full Precision)
model_bnn.add(Dense(len(classes), activation='softmax'))

In [ ]:
# --- Compile BNN Model ---
model_bnn.compile(optimizer=Adam(learning_rate=final_lr),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

print(larq_summary(model_bnn))

In [ ]:
# --- Train BNN Model ---
history_bnn = model_bnn.fit(
    X_train, y_train,
    epochs=final_epochs,
    batch_size=final_batch_size,
    validation_data=(X_val, y_val),
    callbacks=[final_early_stopping],
    verbose=1
)

### Evaluate

In [ ]:
# --- Evaluate BNN Model ---
result_all_models_FC, confusion_matrix_FC, predictions_FC = evaluate_NN_models(
    [model_bnn], X_test, y_test, bin_class=False
)


In [ ]:
headers=['models','accuracy','precision','recall','f1_score']
data_normal_FC=[["model"]]
data_normal_FC.extend(result_all_models_FC)
print_table(data_normal_FC, headers)

In [ ]:
draw_confusion_matrix(confusion_matrix_FC,"Confusion Matrix")

In [ ]:
plt.figure()
plt.plot(history_bnn.history['val_accuracy'], color='orange', label='Validation Accuracy')
plt.xlabel("epochs")
plt.ylabel("accuracy")
plt.show()

## Save

In [ ]:
# Convert our Keras model to a TFLite flatbuffer file
with open("SC_model.tflite", "wb") as flatbuffer_file:
    flatbuffer_bytes = lce.convert_keras_model(model_bnn)
    flatbuffer_file.write(flatbuffer_bytes)

## Explainability - SHAP Analysis

In [ ]:
import sys
sys.path.insert(0, '.')
from explainability_analysis import run_multi


os.makedirs('inputs/SC', exist_ok=True)
np.save('inputs/SC/SC_test_X.npy', X_test)
np.save('inputs/SC/SC_test_y.npy', y_test)

sample_indices = list(range(min(3, len(X_test))))
run_multi(
    data_dir='inputs',
    model_dir='.',
    out_dir='explainability_output/SC',
    task='SC',
    indices=sample_indices,
    n_shap_bg=30
)

print(f"\n✓ SHAP analysis completed for SC samples {sample_indices}")
print(f"  Outputs saved to: explainability_output/SC/")
